# Figure 4 — What makes family members bind differently: KLF/SP vs FOX in HepG2

For 15 KLF/SP and 15 FOX BPNet models in HepG2, over the union of their peaks (central 1 kb): observed counts,
predicted counts (no control track), and predictions after knocking out the family's own ('direct') motif instances,
the other motif instances, or all of them (`pipeline/06_family_ko/family_ko.py`).

Inputs: `data/{klf,fox}_obs.parquet`, `data/{klf,fox}_pred.parquet`, `data/{klf,fox}_pairwise_corr_famdirect.csv`.
Model names are `{TF}_HepG2_{BPNet annotation accession}`. No GPU needed.

In [ ]:
import sys
sys.path.insert(0, '../modules')        # run from notebooks/
import paths                            # data root, genome, repo locations (env BPNET_DATA, MISHA_HG38)
from plot_setup import fig_path         # editable PDF text; figures/ in the repository
%config InlineBackend.figure_format = 'retina'

In [ ]:
import pandas as pd
from pypalettes import load_cmap
from plotnine import *
from plotnine_extensions import ggcorrplot, theme_cowplot

cmap = load_cmap('Emrld', cmap_type='continuous')
fun = load_cmap('Acanthurus_olivaceus')

## Observed (lower triangle) and predicted (upper triangle) correlation between family members
Diagonal: each member's observed vs predicted correlation.

In [ ]:
def corr_panel(fam, select):
    obs = pd.read_parquet(paths.DATA / f'{fam}_obs.parquet')[select]
    pred = pd.read_parquet(paths.DATA / f'{fam}_pred.parquet')[select]
    return (
        ggcorrplot(obs.corr(), upper=pred.corr(), diag=obs.corrwith(pred), layout='matrix', lab=True, lab_size=12)
        + theme_cowplot()
        + theme(figure_size=(12, 8), axis_text_x=element_text(rotation=90, ha='right'))
        + scale_fill_gradientn(colors=cmap.hex, limits=(0, 1), name='Corr')
        + guides(fill=guide_colorbar(display='rectangles', nbin=100))
    )

In [ ]:
p = corr_panel('klf', ['SP1_HepG2_ENCSR981BPZ', 'SP2_HepG2_ENCSR052HWR', 'SP4_HepG2_ENCSR715EEU',
                       'SP5_HepG2_ENCSR773LDV', 'KLF16_HepG2_ENCSR911FYQ'])
p.show()
p.save(fig_path('klf_corr_obs_pred.pdf'))

In [ ]:
p = corr_panel('fox', ['FOXA1_HepG2_ENCSR210RPB', 'FOXA2_HepG2_ENCSR441PYW', 'FOXA3_HepG2_ENCSR743PLW',
                       'FOXC1_HepG2_ENCSR710SMT', 'FOXJ3_HepG2_ENCSR117OOQ'])
p.show()
p.save(fig_path('fox_corr_obs_pred.pdf'))

## Pairwise correlations between all members: observed and after each knockout, vs predicted (WT)
One point per pair of members (105 per family).

In [ ]:
comps = pd.concat([pd.read_csv(paths.DATA / 'klf_pairwise_corr_famdirect.csv').assign(fam='KLF'),
                   pd.read_csv(paths.DATA / 'fox_pairwise_corr_famdirect.csv').assign(fam='FOX')])

yvars = ['observed', 'nondirect', 'direct', 'all']
long = comps.melt(id_vars=['level_0', 'level_1', 'fam', 'predicted'], value_vars=yvars,
                  var_name='metric', value_name='value')
long['metric'] = pd.Categorical(long['metric'], categories=yvars)

cors = (long.groupby(['metric', 'fam'], observed=True)
            .apply(lambda d: pd.Series({'r': d['predicted'].corr(d['value']),
                                        'x': d['predicted'].min(), 'y': d['value'].max()}))
            .reset_index())
cors['label'] = 'r = ' + cors['r'].map('{:.2f}'.format)

all_comp = (
    ggplot(long, aes('predicted', 'value', color='fam'))
    + geom_point(aes(fill='fam'), color='black', stroke=0.3, size=4)
    + geom_smooth(method='lm', se=True, fullrange=True)
    + geom_text(aes('x', 'y', label='label'), data=cors, ha='left', va='top', size=10, color='black',
                show_legend=False)
    + scale_color_manual(values=fun.hex)
    + scale_fill_manual(values=fun.hex)
    + facet_grid('metric ~ fam', scales='free')
    + labs(y='')
    + theme_cowplot()
    + theme(figure_size=(10, 20), legend_position='none')
)
all_comp.show()
all_comp.save(fig_path('obs_pred_mut_scatters.pdf'))